In [4]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.metrics import average_precision_score
from sklearn.impute import SimpleImputer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [5]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-12-02 10:42:20.877597-05:00


#### 6.1: Baseline LightGBM with 5-fold CV (PR-AUC + Precision@K)

In [6]:
import lightgbm as lgb

# Load aligned provider train and folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify ID + label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"

assert y_col in df.columns, "Label column not found in mart.provider_train"
assert {"cv_fold", id_col}.issubset(
    folds.columns), "provider_cv_folds missing keys"

# Join folds
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")
assert df["cv_fold"].notna().all(), "Some providers missing cv_fold"

# Features: drop ID/label/fold; numeric only
drop_cols = {id_col, y_col, "cv_fold"}
X = df.drop(columns=list(drop_cols)).apply(pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

# Impute medians
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Class imbalance weighting
pos = int((y == 1).sum())
neg = int((y == 0).sum())
scale_pos_weight = neg / max(1, pos)
print({"n_providers": len(y), "positives": pos, "negatives": neg,
       "scale_pos_weight": round(scale_pos_weight, 2)})

# Metrics


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]   # adjust to your review budget

# Model params (conservative baseline)
lgb_params = dict(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

# Cross-validation
oof_pred = np.zeros(len(y))
fold_metrics = []

for k in sorted(df["cv_fold"].unique()):
    tr_idx = df.index[df["cv_fold"] != k]
    va_idx = df.index[df["cv_fold"] == k]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_imp.iloc[va_idx], y.iloc[va_idx])],
        # use callbacks for logging + early stopping (version-safe)
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0)
        ]
    )

    p = model.predict_proba(
        X_imp.iloc[va_idx], num_iteration=model.best_iteration_)[:, 1]
    oof_pred[va_idx] = p

    ap = average_precision_score(y.iloc[va_idx], p)
    p_at_k = {
        f"p@{K}": precision_at_k(y.iloc[va_idx].values, p, K) for K in K_LIST}
    fold_metrics.append({"fold": int(k), "AP": ap, **p_at_k})

# Aggregate metrics
fold_df = pd.DataFrame(fold_metrics)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {col: fold_df[col].mean()
                for col in fold_df.columns if col.startswith("p@")}

print("\nFold metrics:")
print(fold_df.to_string(index=False))
print("\nOOF summary:")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
      **{k: round(v, 4) for k, v in p_at_k_means.items()}})

# Persist OOF predictions for audit
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof_pred})


with engine.begin() as conn:
    oof_df.to_sql(
        "provider_oof_predictions_lightgbm",
        con=conn, schema="mart",
        index=False, if_exists="replace", method="multi"
    )

print("\nSaved: mart.provider_oof_predictions_lightgbm")

{'n_providers': 5410, 'positives': 506, 'negatives': 4904, 'scale_pos_weight': 9.69}
[LightGBM] [Info] Number of positive: 405, number of negative: 3923
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002017 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5032
[LightGBM] [Info] Number of data points in the train set: 4328, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.093577 -> initscore=-2.270725
[LightGBM] [Info] Start training from score -2.270725
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[23]	valid_0's binary_logloss: 0.207177
[LightGBM] [Info] Number of positive: 405, number of negative: 3923
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5031
[LightGBM] [Info] Number 

ValueError: Table 'provider_oof_predictions' already exists.

#### 6.2 — LightGBM quick tuning

In [7]:
import itertools
import random
import datetime as dt
from sklearn.metrics import average_precision_score

# Load aligned train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Class weight
pos = int((y == 1).sum())
neg = int((y == 0).sum())
scale_pos_weight = neg / max(1, pos)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
fold_ids = sorted(df["cv_fold"].unique())

# Baseline params kept fixed
base = dict(
    objective="binary",
    n_estimators=3000,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

# Small, sensible search space (guided by LightGBM docs)
grid = {
    "learning_rate":   [0.03, 0.05, 0.08],
    "num_leaves":      [15, 31, 63],
    "min_data_in_leaf": [20, 50, 100, 200],
    "feature_fraction": [0.6, 0.8, 1.0],
    "bagging_fraction": [0.6, 0.8, 1.0],
    "lambda_l1":       [0.0, 1.0, 5.0],
    "lambda_l2":       [0.0, 1.0, 5.0],
    "min_gain_to_split": [0.0, 0.1]
}

# Draw a compact randomized set (e.g., 40 tries)
random.seed(42)
keys, vals = zip(*grid.items())
tries = 40
samples = []
while len(samples) < tries:
    samples.append(dict(zip(keys, [random.choice(v) for v in vals])))


def cv_score(params):
    # Merge base + trial
    params = {**base, **params}
    oof = np.zeros(len(y))
    fold_rows = []
    for k in fold_ids:
        tr = df.index[df["cv_fold"] != k]
        va = df.index[df["cv_fold"] == k]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_imp.iloc[tr], y.iloc[tr],
            eval_set=[(X_imp.iloc[va], y.iloc[va])],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=0)
            ]
        )
        p = model.predict_proba(
            X_imp.iloc[va], num_iteration=model.best_iteration_)[:, 1]
        oof[va] = p

        ap = average_precision_score(y.iloc[va], p)
        row = {"fold": int(k), "AP": ap}
        for K in K_LIST:
            row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    summary = {"AP_mean": fold_df["AP"].mean(), "AP_std": fold_df["AP"].std()}
    for K in K_LIST:
        summary[f"p@{K}"] = fold_df[f"p@{K}"].mean()

    return summary, oof


best = None
best_params = None
best_oof = None

for i, trial in enumerate(samples, 1):
    summary, oof = cv_score(trial)
    score_tuple = (summary["AP_mean"], summary["p@100"])   # tie-break on p@100
    if (best is None) or (score_tuple > (best["AP_mean"], best["p@100"])):
        best, best_params, best_oof = summary, trial, oof
    print(
        f"try {i:02d}/{tries}  AP={summary['AP_mean']:.4f}  p@50={summary['p@50']:.3f}  p@100={summary['p@100']:.3f}")

print("\nBest params:", best_params)
print("Best CV summary:", {k: round(v, 4) for k, v in best.items()})

# Save the winning OOF for audit
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": best_oof})


with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS mart.provider_oof_predictions"))
    oof_df.to_sql("provider_oof_predictions",
                  con=engine, schema="mart", index=False, if_exists="replace")
print("\nSaved: mart.provider_oof_predictions (tuned)")

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] lambda_l1 is set=0.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set=0.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split

KeyboardInterrupt: 

#### 6.3 — Final model + SHAP explanations

In [11]:
# ---- Step 6.3 (fixed): Final LightGBM + SHAP ----
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
import lightgbm as lgb
import shap

# 1) Load train matrix
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)

# Identify ID + label
id_col_candidates = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")]
assert len(id_col_candidates) >= 1, "No provider ID column found."
id_col = id_col_candidates[0]
y_col = "label_provider_fraud_1_0"
assert y_col in df.columns, "Label column not found in mart.provider_train."

# 2) Features / label
X = df.drop(columns=[id_col, y_col]).apply(pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

# 3) Impute
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# 4) Best params from Step 6.2 (your printed best)
best_params = dict(
    objective="binary",
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=50,
    feature_fraction=1.0,
    bagging_fraction=0.6,
    lambda_l1=0.0,
    lambda_l2=0.0,
    min_gain_to_split=0.0,
    n_estimators=3000,
    random_state=42,
    n_jobs=-1,
)

# class weight (neg/pos ≈ 9.69)
pos = int((y == 1).sum())
neg = int((y == 0).sum())
best_params["scale_pos_weight"] = neg / max(1, pos)

# 5) Small validation split for early stopping
X_tr, X_va, y_tr, y_va, id_tr, id_va = train_test_split(
    X_imp, y, df[id_col], test_size=0.15, stratify=y, random_state=42
)

model = lgb.LGBMClassifier(**best_params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150),
        lgb.log_evaluation(period=0)
    ]
)

print({"best_iteration_": model.best_iteration_})

# 6) Score all providers with the final model
y_score = model.predict_proba(X_imp, num_iteration=model.best_iteration_)[:, 1]
scores_df = pd.DataFrame({id_col: df[id_col], "y_score": y_score})

# 7) SHAP explanations (TreeExplainer for LightGBM)
explainer = shap.TreeExplainer(model)
try:
    explanation = explainer(X_imp)
    sv = explanation.values
except Exception:
    sv = explainer.shap_values(X_imp)
    if isinstance(sv, list):
        sv = sv[1]

# 8) Global importance: mean(|SHAP|)
mean_abs = np.abs(sv).mean(axis=0)
global_imp = pd.DataFrame({
    "feature": X_imp.columns,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False)

print("\nTop 15 global features by |SHAP|:")
print(global_imp.head(15).to_string(index=False))

# 9) Per-provider top factors (top 5 by |shap|)
top_k = 5
abs_sv = np.abs(sv)
top_idx = np.argsort(-abs_sv, axis=1)[:, :top_k]

rows = []
for i in range(X_imp.shape[0]):
    prov = df[id_col].iloc[i]
    for rank, j in enumerate(top_idx[i], start=1):
        rows.append({
            id_col: prov,
            "rank": rank,
            "feature": X_imp.columns[j],
            "shap_value": float(sv[i, j]),
            "abs_shap": float(abs_sv[i, j]),
            "direction": "up" if sv[i, j] > 0 else "down"
        })
top_df = pd.DataFrame(rows)

# 10) Persist artifacts — pass a SQLAlchemy Engine/Connection
with engine.begin() as conn:
    scores_df.to_sql("provider_scores_final", con=conn, schema="mart",
                     index=False, if_exists="replace", method="multi")
    global_imp.to_sql("provider_shap_global", con=conn, schema="mart",
                      index=False, if_exists="replace", method="multi")
    top_df.to_sql("provider_shap_top", con=conn, schema="mart",
                  index=False, if_exists="replace", method="multi")

# 11) Quick verifications
with engine.connect() as conn:
    n_scores = pd.read_sql(
        "SELECT COUNT(*) AS c FROM mart.provider_scores_final", con=conn)["c"].iloc[0]
    n_global = pd.read_sql(
        "SELECT COUNT(*) AS c FROM mart.provider_shap_global", con=conn)["c"].iloc[0]
    n_top = pd.read_sql(
        "SELECT COUNT(*) AS c FROM mart.provider_shap_top", con=conn)["c"].iloc[0]

print({"rows_scores": n_scores, "rows_global": n_global, "rows_top": n_top})

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] lambda_l1 is set=0.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set=0.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split

#### 6.4 — Finalize & score the test set

In [13]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# 1) Load train/test
train = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
test = pd.read_sql("SELECT * FROM mart.provider_test",  con=engine)

# Identify ID + label
id_col = [c for c in train.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"

# 2) Build X/y and imputer from TRAIN ONLY
X_tr_full = train.drop(columns=[id_col, y_col]).apply(
    pd.to_numeric, errors="coerce")
y_tr_full = train[y_col].astype(int)
imp = SimpleImputer(strategy="median")
X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr_full),
                        columns=X_tr_full.columns, index=X_tr_full.index)

# 3) Validation split for early stopping
X_t, X_v, y_t, y_v = train_test_split(
    X_tr_imp, y_tr_full, test_size=0.15, stratify=y_tr_full, random_state=42)

# 4) Best params from 6.2 (keep your values)
best_params = dict(
    objective="binary",
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=50,
    feature_fraction=1.0,
    bagging_fraction=0.6,
    lambda_l1=0.0,
    lambda_l2=0.0,
    min_gain_to_split=0.0,
    n_estimators=3000,
    random_state=42,
    n_jobs=-1,
)
# class weight from train distribution
pos = int((y_tr_full == 1).sum())
neg = int((y_tr_full == 0).sum())
best_params["scale_pos_weight"] = neg / max(1, pos)

# 5) Fit final model
clf = lgb.LGBMClassifier(**best_params)
clf.fit(
    X_t, y_t,
    eval_set=[(X_v, y_v)],
    callbacks=[lgb.early_stopping(
        stopping_rounds=150), lgb.log_evaluation(period=0)]
)
print({"best_iteration_": clf.best_iteration_})

# 6) Score TEST
X_te = test.drop(columns=[id_col]).apply(pd.to_numeric, errors="coerce")
X_te_imp = pd.DataFrame(imp.transform(
    X_te), columns=X_te.columns, index=X_te.index)  # reuse TRAIN imputer
test_scores = pd.DataFrame({
    id_col: test[id_col],
    "y_score": clf.predict_proba(X_te_imp, num_iteration=clf.best_iteration_)[:, 1]
})

# 7) Save to DB
with engine.begin() as conn:
    test_scores.to_sql("provider_scores", con=conn, schema="test",
                       index=False, if_exists="replace", method="multi")

# 8) Quick verification
with engine.connect() as conn:
    n_test = pd.read_sql(
        "SELECT COUNT(*) AS c FROM test.provider_scores", con=conn)["c"].iloc[0]
print({"rows_test_scored": n_test})

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] lambda_l1 is set=0.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set=0.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split

#### 6.5 (A): Random Forest baseline — 5-fold CV (AP + Precision@K)

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# Load train and folds (same as before)
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify id/label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

# Features
X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]

# RF with class_weight='balanced' for imbalance
rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"  # handle imbalance per sklearn docs
)

oof = np.zeros(len(y))
fold_rows = []
for k in sorted(df["cv_fold"].unique()):
    tr = df.index[df["cv_fold"] != k]
    va = df.index[df["cv_fold"] == k]

    rf.fit(X_imp.iloc[tr], y.iloc[tr])
    p = rf.predict_proba(X_imp.iloc[va])[:, 1]
    oof[va] = p

    ap = average_precision_score(y.iloc[va], p)
    row = {"fold": int(k), "AP": ap}
    for K in K_LIST:
        row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
    fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {col: fold_df[col].mean()
                for col in fold_df.columns if col.startswith("p@")}

print("Fold metrics (RF):")
print(fold_df.to_string(index=False))
print("\nOOF summary (RF):")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
      **{k: round(v, 4) for k, v in p_at_k_means.items()}})

# Save OOF scores for later ensembling/thresholding
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof})
with engine.begin() as conn:
    oof_df.to_sql("provider_oof_predictions_rf", con=conn,
                  schema="mart", index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_rf")

Fold metrics (RF):
 fold   AP  p@50  p@100  p@200
    0 0.66  0.80   0.62   0.41
    1 0.74  0.86   0.68   0.45
    2 0.68  0.80   0.61   0.42
    3 0.70  0.78   0.61   0.46
    4 0.66  0.78   0.59   0.40

OOF summary (RF):
{'AP_mean': np.float64(0.6897), 'AP_std': np.float64(0.0362), 'p@50': np.float64(0.804), 'p@100': np.float64(0.622), 'p@200': np.float64(0.425)}

Saved: mart.provider_oof_predictions_rf


##### 6.5B: XGBoost CV with AUCPR (AP + Precision@K)

In [19]:
# ---- Step 6.5B (fixed): XGBoost CV with AUCPR (AP + Precision@K), version-safe ----
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sqlalchemy import text
import xgboost as xgb

# Load train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify id/label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

# Features
X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Imbalance ratio (neg/pos)
pos = int((y == 1).sum())
neg = int((y == 0).sum())
scale_pos_weight = neg / max(1, pos)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
fold_ids = sorted(df["cv_fold"].unique())

# Base params shared
base_params = dict(
    objective="binary:logistic",
    n_estimators=3000,       # early stopping will select best iteration
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

oof = np.zeros(len(y))
fold_rows = []

for k in fold_ids:
    tr = df.index[df["cv_fold"] != k]
    va = df.index[df["cv_fold"] == k]

    # Try constructor-style early stopping (newer XGBoost >= 2.1)
    try:
        clf = xgb.XGBClassifier(
            **base_params,
            eval_metric="aucpr",            # set in constructor (>=2.1)
            early_stopping_rounds=150,      # set in constructor (>=2.1)
            verbosity=0
        )
        clf.fit(X_imp.iloc[tr], y.iloc[tr], eval_set=[
                (X_imp.iloc[va], y.iloc[va])])
    except TypeError:
        # Fallback for older XGBoost: early_stopping_rounds in fit()
        clf = xgb.XGBClassifier(
            **base_params,
            verbosity=0
        )
        clf.fit(
            X_imp.iloc[tr], y.iloc[tr],
            eval_set=[(X_imp.iloc[va], y.iloc[va])],
            eval_metric="aucpr",
            early_stopping_rounds=150
        )

    # With early stopping, sklearn API predict/predict_proba use best_iteration automatically. :contentReference[oaicite:2]{index=2}
    p = clf.predict_proba(X_imp.iloc[va])[:, 1]
    oof[va] = p

    ap = average_precision_score(y.iloc[va], p)
    row = {"fold": int(k), "AP": ap}
    for K in K_LIST:
        row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
    fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {col: fold_df[col].mean()
                for col in fold_df.columns if col.startswith("p@")}

print("Fold metrics (XGB, version-safe):")
print(fold_df.to_string(index=False))
print("\nOOF summary (XGB):")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
      **{k: round(v, 4) for k, v in p_at_k_means.items()}})

# Save OOF scores for later compare/ensemble
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof})
with engine.begin() as conn:
    oof_df.to_sql("provider_oof_predictions_xgb", con=conn, schema="mart",
                  index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_xgb")

[0]	validation_0-aucpr:0.50256
[1]	validation_0-aucpr:0.58823
[2]	validation_0-aucpr:0.59921
[3]	validation_0-aucpr:0.60267
[4]	validation_0-aucpr:0.60360
[5]	validation_0-aucpr:0.60316
[6]	validation_0-aucpr:0.60477
[7]	validation_0-aucpr:0.60864
[8]	validation_0-aucpr:0.61127
[9]	validation_0-aucpr:0.60715
[10]	validation_0-aucpr:0.60731
[11]	validation_0-aucpr:0.60920
[12]	validation_0-aucpr:0.60530
[13]	validation_0-aucpr:0.60422
[14]	validation_0-aucpr:0.60901
[15]	validation_0-aucpr:0.61004
[16]	validation_0-aucpr:0.62306
[17]	validation_0-aucpr:0.62232
[18]	validation_0-aucpr:0.62104
[19]	validation_0-aucpr:0.62908
[20]	validation_0-aucpr:0.62888
[21]	validation_0-aucpr:0.62913
[22]	validation_0-aucpr:0.62972
[23]	validation_0-aucpr:0.64313
[24]	validation_0-aucpr:0.65056
[25]	validation_0-aucpr:0.65273
[26]	validation_0-aucpr:0.65911
[27]	validation_0-aucpr:0.65843
[28]	validation_0-aucpr:0.65851
[29]	validation_0-aucpr:0.66012
[30]	validation_0-aucpr:0.66021
[31]	validation_0-

#### 6.5C — CatBoost CV (AP + Precision@K), imbalanced-aware

In [20]:
from catboost import CatBoostClassifier

# Load train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify id/label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

# Features
X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

# Imbalance (neg/pos) for info
pos = int((y == 1).sum())
neg = int((y == 0).sum())
ratio = neg/max(1, pos)
print({"class_ratio_neg_to_pos": round(ratio, 2)})

def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
fold_ids = sorted(df["cv_fold"].unique())

# CatBoost params (version-safe), use auto_class_weights for imbalance
cb_params = dict(
    loss_function="Logloss",
    eval_metric="PRAUC",             # PR-friendly internal metric
    auto_class_weights="Balanced",   # documented imbalance handling
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=42,
    iterations=3000,
    od_type="Iter",                  # early stopping
    od_wait=150,
    verbose=False
)

oof = np.zeros(len(y))
fold_rows = []

for k in fold_ids:
    tr = df.index[df["cv_fold"] != k]
    va = df.index[df["cv_fold"] == k]

    clf = CatBoostClassifier(**cb_params)
    clf.fit(X_imp.iloc[tr], y.iloc[tr],
            eval_set=(X_imp.iloc[va], y.iloc[va]),
            use_best_model=True)

    p = clf.predict_proba(X_imp.iloc[va])[:, 1]
    oof[va] = p

    ap = average_precision_score(y.iloc[va], p)
    row = {"fold": int(k), "AP": ap}
    for K in K_LIST:
        row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
    fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {c: fold_df[c].mean()
                for c in fold_df.columns if c.startswith("p@")}

print("Fold metrics (CatBoost):")
print(fold_df.to_string(index=False))
print("\nOOF summary (CatBoost):")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
       **{k: round(v, 4) for k, v in p_at_k_means.items()}})

# Save OOF
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof})
with engine.begin() as conn:
    oof_df.to_sql("provider_oof_predictions_cat", con=conn, schema="mart",
                  index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_cat")

{'class_ratio_neg_to_pos': 9.69}
Fold metrics (CatBoost):
 fold   AP  p@50  p@100  p@200
    0 0.70  0.80   0.61   0.41
    1 0.76  0.82   0.70   0.44
    2 0.72  0.86   0.62   0.43
    3 0.73  0.84   0.63   0.45
    4 0.68  0.80   0.60   0.39

OOF summary (CatBoost):
{'AP_mean': np.float64(0.7163), 'AP_std': np.float64(0.0296), 'p@50': np.float64(0.824), 'p@100': np.float64(0.632), 'p@200': np.float64(0.423)}

Saved: mart.provider_oof_predictions_cat


#### 6.5D — ExtraTrees CV (AP + Precision@K)

In [21]:
from sklearn.ensemble import ExtraTreesClassifier

df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
fold_ids = sorted(df["cv_fold"].unique())

# ExtraTrees: very randomized trees; class_weight='balanced' helps on imbalance
et = ExtraTreesClassifier(
    n_estimators=1200,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

oof = np.zeros(len(y))
fold_rows = []
for k in fold_ids:
    tr = df.index[df["cv_fold"] != k]
    va = df.index[df["cv_fold"] == k]

    et.fit(X_imp.iloc[tr], y.iloc[tr])
    p = et.predict_proba(X_imp.iloc[va])[:, 1]
    oof[va] = p

    ap = average_precision_score(y.iloc[va], p)
    row = {"fold": int(k), "AP": ap}
    for K in K_LIST:
        row[f"p@{K}"] = precision_at_k(y.iloc[va].values, p, K)
    fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
ap_mean, ap_std = fold_df["AP"].mean(), fold_df["AP"].std()
p_at_k_means = {c: fold_df[c].mean()
                for c in fold_df.columns if c.startswith("p@")}

print("Fold metrics (ExtraTrees):")
print(fold_df.to_string(index=False))
print("\nOOF summary (ExtraTrees):")
print({"AP_mean": round(ap_mean, 4), "AP_std": round(ap_std, 4),
       **{k: round(v, 4) for k, v in p_at_k_means.items()}})

with engine.begin() as conn:
    pd.DataFrame({id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof}) \
      .to_sql("provider_oof_predictions_et", con=conn, schema="mart",
              index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_et")

Fold metrics (ExtraTrees):
 fold   AP  p@50  p@100  p@200
    0 0.66  0.80   0.61   0.41
    1 0.75  0.86   0.68   0.45
    2 0.71  0.82   0.63   0.43
    3 0.71  0.80   0.57   0.46
    4 0.68  0.80   0.60   0.41

OOF summary (ExtraTrees):
{'AP_mean': np.float64(0.7016), 'AP_std': np.float64(0.035), 'p@50': np.float64(0.816), 'p@100': np.float64(0.618), 'p@200': np.float64(0.43)}

Saved: mart.provider_oof_predictions_et


##### 6.5E — Logistic Regression (elastic-net) with CV (AP + Precision@K)

In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)

# Identify id/label
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

# Features
X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int)

# Impute + standardize
imp = SimpleImputer(strategy="median")
Xs = imp.fit_transform(X)
sc = StandardScaler()
Xs = sc.fit_transform(Xs)


def precision_at_k(y_true, y_score, k):
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))


K_LIST = [50, 100, 200]
oof = np.zeros(len(y))
rows = []

lr = LogisticRegression(
    penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0,
    class_weight="balanced", max_iter=5000, n_jobs=-1, random_state=42
)

for k in sorted(df["cv_fold"].unique()):
    tr = df.index[df["cv_fold"] != k]
    va = df.index[df["cv_fold"] == k]
    lr.fit(Xs[tr], y.iloc[tr])
    p = lr.predict_proba(Xs[va])[:, 1]
    oof[va] = p

    ap = average_precision_score(y.iloc[va], p)
    row = {"fold": int(k), "AP": ap}
    for KK in K_LIST:
        row[f"p@{KK}"] = precision_at_k(y.iloc[va].values, p, KK)
    rows.append(row)

fold_df = pd.DataFrame(rows)
summary = {"AP_mean": fold_df["AP"].mean(), "AP_std": fold_df["AP"].std()}
for KK in K_LIST:
    summary[f"p@{KK}"] = fold_df[f"p@{KK}"].mean()
print("Fold metrics (LogReg):")
print(fold_df.to_string(index=False))
print("\nOOF summary (LogReg):", {k: round(v, 4) for k, v in summary.items()})

# Save OOF
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof})
with engine.begin() as conn:
    oof_df.to_sql("provider_oof_predictions_lr", con=conn, schema="mart",
                  index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_lr")

Fold metrics (LogReg):
 fold   AP  p@50  p@100  p@200
    0 0.66  0.74   0.62   0.42
    1 0.73  0.84   0.67   0.45
    2 0.70  0.82   0.61   0.41
    3 0.73  0.84   0.63   0.46
    4 0.72  0.84   0.62   0.41

OOF summary (LogReg): {'AP_mean': np.float64(0.7096), 'AP_std': np.float64(0.0303), 'p@50': np.float64(0.816), 'p@100': np.float64(0.63), 'p@200': np.float64(0.429)}

Saved: mart.provider_oof_predictions_lr


##### 6.5F — MLP (Neural Network) with CV (AP + Precision@K, PyTorch)

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load train + folds
df = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
folds = pd.read_sql("SELECT * FROM mart.provider_cv_folds", con=engine)
id_col = [c for c in df.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"
df = df.merge(folds[[id_col, "cv_fold"]], on=id_col, how="inner")

# Features: impute + standardize
X = df.drop(columns=[id_col, y_col, "cv_fold"]).apply(
    pd.to_numeric, errors="coerce")
y = df[y_col].astype(int).values
imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
sc = StandardScaler()
X_std = sc.fit_transform(X_imp)

X_tensor = torch.tensor(X_std, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

pos = float((y == 1).sum())
neg = float((y == 0).sum())
pos_weight = torch.tensor([neg/max(1.0, pos)], dtype=torch.float32).to(device)


def precision_at_k(y_true_np, y_score_np, k):
    k = min(k, len(y_true_np))
    idx = np.argsort(-y_score_np)[:k]
    return float(np.mean(y_true_np[idx]))


K_LIST = [50, 100, 200]
oof = np.zeros(len(y))

# Simple MLP


class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.BatchNorm1d(
                128), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 1)  # logits
        )

    def forward(self, x): return self.net(x)


def train_fold(tr_idx, va_idx, max_epochs=100, batch_size=256, patience=10):
    model = MLP(X_tensor.shape[1]).to(device)
    # AdamW per Loshchilov & Hutter
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_ds = TensorDataset(X_tensor[tr_idx], y_tensor[tr_idx])
    valid_ds = TensorDataset(X_tensor[va_idx], y_tensor[va_idx])
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

    best_loss, best_state, no_imp = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()

        # val loss (proxy early stopping)
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in valid_dl:
                xb, yb = xb.to(device), yb.to(device)
                val_losses.append(crit(model(xb), yb).item())
        vloss = float(np.mean(val_losses))

        if vloss + 1e-6 < best_loss:
            best_loss, best_state, no_imp = vloss, {
                k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    # get probabilities on validation
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor[va_idx].to(device)).cpu().numpy().reshape(-1)
        proba = 1.0/(1.0+np.exp(-logits))
    return proba


rows = []
for k in sorted(df["cv_fold"].unique()):
    tr_idx = df.index[df["cv_fold"] != k].to_numpy()
    va_idx = df.index[df["cv_fold"] == k].to_numpy()
    p = train_fold(tr_idx, va_idx)
    oof[va_idx] = p

    ap = average_precision_score(y[va_idx], p)
    row = {"fold": int(k), "AP": ap}
    for KK in K_LIST:
        row[f"p@{KK}"] = precision_at_k(y[va_idx], p, KK)
    rows.append(row)

fold_df = pd.DataFrame(rows)
summary = {"AP_mean": fold_df["AP"].mean(), "AP_std": fold_df["AP"].std()}
for KK in K_LIST:
    summary[f"p@{KK}"] = fold_df[f"p@{KK}"].mean()
print("Fold metrics (MLP):")
print(fold_df.to_string(index=False))
print("\nOOF summary (MLP):", {k: round(v, 4) for k, v in summary.items()})

# Save OOF
oof_df = pd.DataFrame(
    {id_col: df[id_col], "cv_fold": df["cv_fold"], "y_true": y, "y_score": oof})
with engine.begin() as conn:
    oof_df.to_sql("provider_oof_predictions_mlp", con=conn, schema="mart",
                  index=False, if_exists="replace", method="multi")
print("\nSaved: mart.provider_oof_predictions_mlp")

Fold metrics (MLP):
 fold   AP  p@50  p@100  p@200
    0 0.65  0.76   0.62   0.41
    1 0.74  0.82   0.68   0.45
    2 0.71  0.86   0.61   0.41
    3 0.74  0.88   0.63   0.43
    4 0.71  0.82   0.66   0.42

OOF summary (MLP): {'AP_mean': np.float64(0.7097), 'AP_std': np.float64(0.0354), 'p@50': np.float64(0.828), 'p@100': np.float64(0.64), 'p@200': np.float64(0.423)}

Saved: mart.provider_oof_predictions_mlp


##### 6.6 — Leaderboard

In [30]:
def detect_id_col(columns):
    for c in columns:
        if c.lower() in ("provider", "provider_id", "npi", "prov_id"):
            return c
    raise RuntimeError("No provider ID column found.")


def load_oof_table(tbl, engine, std_id="prov_id"):
    df = pd.read_sql(f"SELECT * FROM {tbl}", con=engine)
    id_col = detect_id_col(df.columns)
    assert {"y_true", "y_score", "cv_fold"}.issubset(
        df.columns), f"{tbl} must have y_true,y_score,cv_fold"
    key = tbl.split('.')[-1]
    out = df[[id_col, "cv_fold", "y_true", "y_score"]].rename(
        columns={id_col: std_id, "y_score": key}
    )
    return out, std_id, key


candidate_tables = [
    "mart.provider_oof_predictions",       # LightGBM tuned
    "mart.provider_oof_predictions_rf",
    "mart.provider_oof_predictions_xgb",
    "mart.provider_oof_predictions_cat",
    "mart.provider_oof_predictions_et",
    "mart.provider_oof_predictions_lr",
    "mart.provider_oof_predictions_mlp",
]

loaded = []
for tbl in candidate_tables:
    try:
        dfi, std_id, key = load_oof_table(tbl, engine)
        loaded.append((tbl, dfi, key))
    except Exception:
        pass
if not loaded:
    raise RuntimeError("No OOF tables found.")

base_tbl, base_df, _ = loaded[0]
for _, dfi, _ in loaded[1:]:
    base_df = base_df.merge(dfi, on=[std_id, "cv_fold", "y_true"], how="inner")


def precision_at_k(y, s, K):
    K = min(K, len(y))
    idx = np.argsort(-s)[:K]
    return float(np.mean(y.iloc[idx]))


K_LIST = [50, 100, 200]
score_cols = [c for c in base_df.columns if c.startswith(
    "provider_oof_predictions")]

# ---- (A) Global OOF leaderboard (as before) ----
rows = []
for col in score_cols:
    ap = average_precision_score(base_df["y_true"], base_df[col])
    row = {"model": col, "AP": ap}
    for K in K_LIST:
        row[f"p@{K}"] = precision_at_k(base_df["y_true"], base_df[col], K)
    rows.append(row)
lb_global = pd.DataFrame(rows).sort_values("AP", ascending=False)
print("Leaderboard (Global OOF):")
print(lb_global.to_string(index=False))

# ---- (B) Fold-averaged leaderboard (sanity, comparable to earlier fold tables) ----
rows_f = []
for col in score_cols:
    parts = []
    for f in sorted(base_df["cv_fold"].unique()):
        fold = base_df[base_df["cv_fold"] == f]
        ap_f = average_precision_score(fold["y_true"], fold[col])
        rec = {"fold": int(f), "AP": ap_f}
        for K in K_LIST:
            rec[f"p@{K}"] = precision_at_k(fold["y_true"], fold[col], K)
        parts.append(rec)
    fold_df = pd.DataFrame(parts)
    rows_f.append({
        "model": col,
        "AP_mean": fold_df["AP"].mean(),
        "AP_std":  fold_df["AP"].std(),
        **{f"p@{K}": fold_df[f"p@{K}"].mean() for K in K_LIST}
    })
lb_foldavg = pd.DataFrame(rows_f).sort_values("AP_mean", ascending=False)
print("\nLeaderboard (Fold-averaged OOF):")
print(lb_foldavg.to_string(index=False))

# ---- Ensembles (same three) using GLOBAL ranks/scores ----
w = lb_global.set_index("model")["AP"]
w = w / w.sum()
ranks = np.vstack([pd.Series(base_df[c]).rank(
    pct=True).values for c in score_cols]).T
base_df["ens_rankavg"] = ranks.mean(axis=1)
base_df["ens_rankavg_w"] = (ranks * np.array([w[c]
                            for c in score_cols])).sum(axis=1)
base_df["ens_soft_w"] = np.sum([base_df[c].values * w[c]
                               for c in score_cols], axis=0)


def eval_vec(name, s):
    ap = average_precision_score(base_df["y_true"], s)
    out = {"model": name, "AP": ap}
    for K in K_LIST:
        out[f"p@{K}"] = precision_at_k(base_df["y_true"], s, K)
    return out


ens = [eval_vec("ens_rankavg", base_df["ens_rankavg"]),
       eval_vec("ens_rankavg_w", base_df["ens_rankavg_w"]),
       eval_vec("ens_soft_w", base_df["ens_soft_w"])]
ens_df = pd.DataFrame(ens).sort_values("AP", ascending=False)
print("\nEnsemble OOF (Global):")
print(ens_df.to_string(index=False))

Leaderboard (Global OOF):
                       model   AP  p@50  p@100  p@200
 provider_oof_predictions_lr 0.71  0.94   0.94   0.88
provider_oof_predictions_cat 0.71  0.98   0.97   0.88
provider_oof_predictions_mlp 0.70  0.96   0.96   0.87
 provider_oof_predictions_et 0.70  1.00   0.95   0.87
provider_oof_predictions_xgb 0.69  0.98   0.93   0.86
 provider_oof_predictions_rf 0.69  1.00   0.94   0.84

Leaderboard (Fold-averaged OOF):
                       model  AP_mean  AP_std  p@50  p@100  p@200
provider_oof_predictions_cat     0.72    0.03  0.82   0.63   0.42
provider_oof_predictions_mlp     0.71    0.04  0.83   0.64   0.42
 provider_oof_predictions_lr     0.71    0.03  0.82   0.63   0.43
 provider_oof_predictions_et     0.70    0.04  0.82   0.62   0.43
provider_oof_predictions_xgb     0.70    0.03  0.82   0.63   0.42
 provider_oof_predictions_rf     0.69    0.04  0.80   0.62   0.42

Ensemble OOF (Global):
        model   AP  p@50  p@100  p@200
ens_rankavg_w 0.72  0.98   0.97   0.8

#### 6.7 — Score test, build ensembles, and write labels (Top-K & precision-target)

In [34]:
# ---- Step 6.7 (patched): finalize test labels (Top-K + precision-target) ----
import pandas as pd, numpy as np
from sqlalchemy import text
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier

# ----------------------------
# 0) Settings you can change
# ----------------------------
K = 100                     # Top-K audit budget
TARGET_PRECISION = 0.70     # target precision (choose your policy target)
INCLUDE_MODELS = ["lgbm","xgb","cat","rf","et","lr","mlp"]  # add "mlp" to include the neural net

# ----------------------------
# 1) Load train/test
# ----------------------------
train = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
test  = pd.read_sql("SELECT * FROM mart.provider_test",  con=engine)
id_col = [c for c in train.columns if c.lower() in ("provider","provider_id","npi","prov_id")][0]
y_col  = "label_provider_fraud_1_0"

X_tr = train.drop(columns=[id_col, y_col]).apply(pd.to_numeric, errors="coerce")
y_tr = train[y_col].astype(int)
X_te = test.drop(columns=[id_col]).apply(pd.to_numeric, errors="coerce")

# Imputation for all; scaler for LR/MLP
imp = SimpleImputer(strategy="median")
X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
X_te_imp = pd.DataFrame(imp.transform(X_te), columns=X_te.columns, index=X_te.index)
scaler = StandardScaler()
X_tr_std = pd.DataFrame(scaler.fit_transform(X_tr_imp), columns=X_tr.columns, index=X_tr.index)
X_te_std = pd.DataFrame(scaler.transform(X_te_imp), columns=X_te.columns, index=X_te.index)

pos = int((y_tr==1).sum()); neg = int((y_tr==0).sum())
spw = neg / max(1, pos)

# ----------------------------
# 2) Refit models on TRAIN and score TEST
# ----------------------------
X_t, X_v, y_t, y_v = train_test_split(X_tr_imp, y_tr, test_size=0.15, stratify=y_tr, random_state=42)
scores = {}

# LightGBM (callbacks for early stopping)
if "lgbm" in INCLUDE_MODELS:
    lgbm = lgb.LGBMClassifier(
        objective="binary", learning_rate=0.03, num_leaves=63, min_data_in_leaf=50,
        feature_fraction=1.0, bagging_fraction=0.6, lambda_l1=0.0, lambda_l2=0.0,
        min_gain_to_split=0.0, n_estimators=3000, random_state=42, n_jobs=-1,
        scale_pos_weight=spw
    )
    lgbm.fit(
        X_t, y_t, eval_set=[(X_v, y_v)],
        callbacks=[lgb.early_stopping(stopping_rounds=150), lgb.log_evaluation(period=0)]
    )
    scores["lgbm"] = lgbm.predict_proba(X_te_imp, num_iteration=lgbm.best_iteration_)[:,1]

# XGBoost (VERSION-SAFE: use callback API for early stopping)
if "xgb" in INCLUDE_MODELS:
    xgbm = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=3000, learning_rate=0.05,
        max_depth=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, scale_pos_weight=spw, eval_metric="aucpr",
        early_stopping_rounds=150
    )
    xgbm.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
    scores["xgb"] = xgbm.predict_proba(X_te_imp)[:,1]

# CatBoost (class imbalance + PRAUC metric)
if "cat" in INCLUDE_MODELS:
    cat = CatBoostClassifier(
        loss_function="Logloss", eval_metric="PRAUC", auto_class_weights="Balanced",
        learning_rate=0.05, depth=6, l2_leaf_reg=3.0, random_seed=42,
        iterations=3000, od_type="Iter", od_wait=150, verbose=False
    )
    cat.fit(X_t, y_t, eval_set=(X_v, y_v), use_best_model=True)
    scores["cat"] = cat.predict_proba(X_te_imp)[:,1]

# Random Forest
if "rf" in INCLUDE_MODELS:
    rf = RandomForestClassifier(
        n_estimators=1000, max_depth=None, max_features="sqrt",
        min_samples_leaf=1, n_jobs=-1, random_state=42, class_weight="balanced"
    )
    rf.fit(X_tr_imp, y_tr)
    scores["rf"] = rf.predict_proba(X_te_imp)[:,1]

# ExtraTrees
if "et" in INCLUDE_MODELS:
    et = ExtraTreesClassifier(
        n_estimators=1200, max_depth=None, max_features="sqrt",
        min_samples_leaf=1, n_jobs=-1, random_state=42, class_weight="balanced"
    )
    et.fit(X_tr_imp, y_tr)
    scores["et"] = et.predict_proba(X_te_imp)[:,1]

# Logistic Regression (elastic-net)
if "lr" in INCLUDE_MODELS:
    lr = LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0,
        class_weight="balanced", max_iter=5000, n_jobs=-1, random_state=42
    )
    lr.fit(X_tr_std, y_tr)
    scores["lr"] = lr.predict_proba(X_te_std)[:,1]

# (Optional) MLP using torch — included only if "mlp" in INCLUDE_MODELS and torch is available
if "mlp" in INCLUDE_MODELS:
    try:
        import torch, torch.nn as nn
        from torch.utils.data import TensorDataset, DataLoader
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        X_t_std, X_v_std = X_tr_std.loc[X_t.index].values, X_tr_std.loc[X_v.index].values
        y_t_np, y_v_np = y_t.values.astype(np.float32), y_v.values.astype(np.float32)
        X_te_std_np = X_te_std.values.astype(np.float32)

        class MLP(nn.Module):
            def __init__(self, d):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(d, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.2),
                    nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
                    nn.Linear(64, 1)
                )
            def forward(self, x): return self.net(x)

        def fit_mlp(Xtr, ytr, Xva, yva, epochs=100, patience=10):
            tr_ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32).view(-1,1))
            va_ds = TensorDataset(torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva, dtype=torch.float32).view(-1,1))
            tr_dl = DataLoader(tr_ds, batch_size=256, shuffle=True)
            va_dl = DataLoader(va_ds, batch_size=256, shuffle=False)
            model = MLP(Xtr.shape[1]).to(device)
            pos = (ytr==1).sum(); neg = (ytr==0).sum()
            pos_weight = torch.tensor([neg/max(1,pos)], dtype=torch.float32).to(device)
            crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            opt  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            best, noimp = 1e9, 0; best_state=None
            for _ in range(epochs):
                model.train()
                for xb, yb in tr_dl:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
                # val
                model.eval(); vs=[]
                with torch.no_grad():
                    for xb, yb in va_dl:
                        xb, yb = xb.to(device), yb.to(device)
                        vs.append(crit(model(xb), yb).item())
                v = float(np.mean(vs))
                if v + 1e-6 < best:
                    best, noimp, best_state = v, 0, {k:v.clone() for k,v in model.state_dict().items()}
                else:
                    noimp += 1
                    if noimp >= patience: break
            if best_state is not None: model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                te_logits = model(torch.tensor(X_te_std_np, dtype=torch.float32).to(device)).cpu().numpy().reshape(-1)
                te_proba  = 1/(1+np.exp(-te_logits))
            return te_proba

        scores["mlp"] = fit_mlp(X_t_std, y_t_np, X_v_std, y_v_np)
    except Exception as e:
        print(f"Skipping MLP (torch not available or failed): {e}")

# ----------------------------
# 3) Build ensembles on TEST (weights from OOF AP)
# ----------------------------
def load_oof(tbl):
    df = pd.read_sql(f"SELECT * FROM {tbl}", con=engine)
    idc = [c for c in df.columns if c.lower() in ("provider","provider_id","npi","prov_id")][0]
    return df.rename(columns={idc: "prov_id"})

oof_tables = {
    "lgbm": "mart.provider_oof_predictions",
    "rf":   "mart.provider_oof_predictions_rf",
    "xgb":  "mart.provider_oof_predictions_xgb",
    "cat":  "mart.provider_oof_predictions_cat",
    "et":   "mart.provider_oof_predictions_et",
    "lr":   "mart.provider_oof_predictions_lr",
    "mlp":  "mart.provider_oof_predictions_mlp",
}
aps = {}
for name, tbl in oof_tables.items():
    if name in scores:
        try:
            dfi = load_oof(tbl)
            aps[name] = average_precision_score(dfi["y_true"], dfi["y_score"])
        except Exception:
            pass
w = pd.Series(aps); w = (w / w.sum()) if len(w) else pd.Series(dtype=float)

score_df = pd.DataFrame({id_col: test[id_col]})
for name, s in scores.items(): score_df[name] = s

# Rank-based ensemble (weighted)
rank_cols = []
for name in scores:
    r = pd.Series(score_df[name]).rank(pct=True).values
    score_df[f"rank_{name}"] = r; rank_cols.append(f"rank_{name}")
score_df["ens_rankavg_w"] = np.sum([score_df[f"rank_{m}"].values * w.get(m, 0.0) for m in scores], axis=0)

# Soft (probability) ensemble (weighted)
score_df["ens_soft_w"] = np.sum([score_df[m].values * w.get(m, 0.0) for m in scores], axis=0)

# ----------------------------
# 4) Persist TEST scores
# ----------------------------
with engine.begin() as conn:
    score_df.to_sql("provider_scores_multi", con=conn, schema="test",
                    index=False, if_exists="replace", method="multi")

# ----------------------------
# 5) Decision rules → labels
# ----------------------------
# (A) Top-K by ensemble
score_df = score_df.sort_values("ens_soft_w", ascending=False).reset_index(drop=True)
score_df["label_topk"] = 0
score_df.loc[:K-1, "label_topk"] = 1
topk_out = score_df[[id_col, "ens_soft_w", "ens_rankavg_w", "label_topk"]].copy()
with engine.begin() as conn:
    topk_out.to_sql("provider_labels_topk", con=conn, schema="test",
                    index=False, if_exists="replace", method="multi")

# (B) Precision-target threshold from OOF ensemble PR curve
try:
    oof_ens = pd.read_sql("SELECT y_true, ens_soft_w FROM mart.provider_oof_predictions_ensemble", con=engine)
    prec, rec, th = precision_recall_curve(oof_ens["y_true"].values, oof_ens["ens_soft_w"].values)
    thr = float(th[:-1][np.where(prec[:-1] >= TARGET_PRECISION)[0][0]]) if (prec[:-1] >= TARGET_PRECISION).any() else float(th.max())
    score_df["label_thresh"] = (score_df["ens_soft_w"] >= thr).astype(int)
    thresh_out = score_df[[id_col, "ens_soft_w", "label_thresh"]].copy()
    with engine.begin() as conn:
        thresh_out.to_sql("provider_labels_thresh", con=conn, schema="test",
                          index=False, if_exists="replace", method="multi")
    print({"precision_target": TARGET_PRECISION, "chosen_threshold": round(thr, 6)})
except Exception as e:
    print(f"Precision-target labeling skipped (need mart.provider_oof_predictions_ensemble): {e}")

# ----------------------------
# 6) Quick checks
# ----------------------------
print({
    "n_test": len(test),
    "topk_K": K,
    "topk_positives": int(topk_out["label_topk"].sum()),
    "models_used": list(scores.keys()),
    "weights_used": {m: round(float(w.get(m, 0.0)), 4) for m in scores}
})


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] lambda_l1 is set=0.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set=0.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] bagging_fraction is set=0.6, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split